# VLM-LiDAR-Camera-ADAS-Perception

I built this project to explore a question that kept coming up during my PhD work on ADAS perception at Infineon: **can Vision Language Models replace the tedious annotate-train-validate cycle** for understanding driving scenes?

Traditional detection models (YOLO, PointPillars) need thousands of labeled images per object class. Every time you encounter a new scenario — construction zones, unusual vehicles, animals on rural roads — you're back to square one with annotations. VLMs flip this entirely. Feed them a raw image, ask a question in plain English, and they reason about the scene using knowledge from billions of training examples.

This notebook walks through the full pipeline I built: loading a quantized LLaVA model, analyzing KITTI driving scenes, fusing LiDAR depth data onto camera images, and generating structured ADAS perception output — all without a single annotation.

**What you need:** A free Colab T4 GPU (Runtime > Change runtime type > T4). That's it.

## Setup

Cloning the repo and installing dependencies. The model download (~4GB) happens later when we first load it.

In [ ]:
!git clone https://github.com/VasuTammisetti/VLM-LiDAR-Camera-ADAS-perception.git
%cd VLM-LiDAR-Camera-ADAS-perception

In [ ]:
!pip install -q torch torchvision
!pip install -q transformers>=4.40 accelerate>=0.27 bitsandbytes>=0.43
!pip install -q Pillow matplotlib numpy sentencepiece protobuf

In [ ]:
# Quick sanity check — make sure the GPU is actually there
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU: {gpu_name} | VRAM: {vram:.1f} GB")
    print("Good to go.")
else:
    print("No GPU found. Go to Runtime > Change runtime type > T4 GPU")
    print("Without a GPU this will be painfully slow.")

## Loading the KITTI Data

I'm using the KITTI Vision Benchmark dataset — 101 training frames with left camera images, Velodyne LiDAR point clouds, and calibration files. My data is structured as:

```
sensorfusion/sensorfusion/
    data_object_image_2/training/     <- 101 PNG images
    data_object_velodyne/training/    <- 101 BIN point clouds  
    data_object_calib/training/       <- 101 TXT calibration files
```

The repo ships with 5 sample images for quick testing. If you have your own KITTI data, upload the zip and unzip it to `/content/`.

In [ ]:
import os
import glob
from PIL import Image
import matplotlib.pyplot as plt

# Check what data we have available
FULL_DATA = "/content/sensorfusion/sensorfusion"
full_img_dir = os.path.join(FULL_DATA, "data_object_image_2/training")
full_vel_dir = os.path.join(FULL_DATA, "data_object_velodyne/training")
full_cal_dir = os.path.join(FULL_DATA, "data_object_calib/training")

sample_dir = "data/sample_scenes"

if os.path.exists(full_img_dir):
    IMAGE_DIR = full_img_dir
    VELODYNE_DIR = full_vel_dir
    CALIB_DIR = full_cal_dir
    has_lidar = os.path.exists(VELODYNE_DIR) and len(os.listdir(VELODYNE_DIR)) > 0
    has_calib = os.path.exists(CALIB_DIR) and len(os.listdir(CALIB_DIR)) > 0
    print(f"Full KITTI dataset found.")
    print(f"  Images:   {len(os.listdir(IMAGE_DIR))} frames")
    print(f"  Velodyne: {len(os.listdir(VELODYNE_DIR))} point clouds")
    print(f"  Calib:    {len(os.listdir(CALIB_DIR))} files")
else:
    IMAGE_DIR = sample_dir
    has_lidar = False
    has_calib = False
    print(f"Using {len(os.listdir(sample_dir))} sample images from repo.")
    print("For LiDAR fusion, upload sensorfusion.zip and unzip to /content/")

In [ ]:
# Quick look at what we're working with
images = sorted(glob.glob(os.path.join(IMAGE_DIR, "*.png")))[:5]

fig, axes = plt.subplots(1, len(images), figsize=(20, 4))
if len(images) == 1:
    axes = [axes]
for ax, path in zip(axes, images):
    ax.imshow(Image.open(path))
    ax.set_title(os.path.basename(path), fontsize=9)
    ax.axis('off')
plt.suptitle('Sample Driving Scenes from KITTI', fontsize=13)
plt.tight_layout()
plt.show()

## Loading the VLM

I'm using **LLaVA-1.6-Mistral-7B** — a 7B parameter vision-language model that handles conversational prompts well. The key trick is **4-bit NF4 quantization** via bitsandbytes, which compresses a 14GB model into roughly 5GB of VRAM. That's what makes this runnable on a free Colab T4.

I initially tried PaliGemma-3B (smaller, faster) but it only understands short commands like `caption en` and couldn't handle the structured ADAS prompts I designed. It kept responding with "Sorry, as a base VLM I am not trained to answer this question." LLaVA handles detailed instructions naturally.

First run downloads the weights (~4GB). After that it uses the cached version.

In [ ]:
from src.model_loader import load_model

model, processor = load_model("llava-1.5-7b")

vram_used = torch.cuda.memory_allocated() / 1024**3
vram_total = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"\nVRAM usage: {vram_used:.1f} / {vram_total:.1f} GB — plenty of headroom for inference.")

## Scene Analysis: RGB Only

This is the core of the project. I feed a raw driving image to the VLM along with a carefully crafted ADAS prompt, and it returns a structured analysis: scene context, detected objects with positions, hazard assessment, and a driving recommendation.

No bounding boxes, no class labels, no post-processing — just natural language scene understanding. The prompt engineering took the most iteration to get right. Specifying the exact output structure and telling the model to "be precise and safety-focused" made a big difference in output quality.

In [ ]:
from src.scene_analyzer import analyze_scene, PROMPTS
from src.visualization import display_results

image_path = sorted(glob.glob(os.path.join(IMAGE_DIR, "*.png")))[0]
image = Image.open(image_path).convert("RGB")

print(f"Analyzing: {os.path.basename(image_path)}")
print(f"Image size: {image.size}")
print("\nRunning full scene analysis...\n")

result = analyze_scene(image, model, processor, prompt_type="full_analysis")
print(result)

display_results(image, result, title=f"Scene: {os.path.basename(image_path)}")

## Different Prompts, Same Image

One of the real advantages over traditional detectors: I can ask completely different questions about the same scene without retraining anything. Here I run three specialized prompts on one image:

- **full_analysis** — complete scene breakdown with all four sections
- **hazard_only** — focused purely on risks, severity levels, and recommended actions
- **object_count** — exhaustive enumeration of every visible object

With YOLO, you'd need three separate models or three training runs. Here it's just a different text prompt.

In [ ]:
image = Image.open(image_path).convert("RGB")

for ptype in ["full_analysis", "hazard_only", "object_count"]:
    print(f"\n{'─' * 60}")
    print(f"  Prompt: {ptype}")
    print(f"{'─' * 60}\n")
    
    result = analyze_scene(image, model, processor, prompt_type=ptype)
    print(result)
    torch.cuda.empty_cache()

## Batch Analysis Across Multiple Scenes

Running the analysis across several scenes to see how consistent the model is. What I found interesting is that it genuinely adapts — urban scenes get mentions of pedestrians and traffic signals, while highway scenes focus on vehicle speeds and lane positions. It's not just templating.

In [ ]:
from src.scene_analyzer import batch_analyze

all_images = sorted(glob.glob(os.path.join(IMAGE_DIR, "*.png")))[:5]
print(f"Analyzing {len(all_images)} scenes...\n")

results = batch_analyze(all_images, model, processor, prompt_type="full_analysis")

for img_path, analysis in results.items():
    image = Image.open(img_path).convert("RGB")
    display_results(image, analysis, title=f"Scene: {os.path.basename(img_path)}")
    torch.cuda.empty_cache()

## Camera-LiDAR Fusion

This is where it gets really interesting. The RGB-only analysis works, but it has no genuine sense of distance — it's guessing based on perspective cues. By projecting Velodyne 64-beam LiDAR points onto the camera image using KITTI's calibration matrices (P2 x R0_rect x Tr_velo_to_cam), I create a depth-colored overlay.

The color encoding: **blue = close (0-10m), green = mid (10-25m), red = far (25-50m)**.

What surprised me is that the VLM actually picks up on these depth cues and produces noticeably more precise distance estimates in its analysis. It's a simple trick — no architecture changes, just a richer visual input.

**Requires:** Full KITTI dataset with Velodyne (.bin) and calibration (.txt) files.

In [ ]:
from src.visualization import create_lidar_overlay, create_combined_view, create_bev
import numpy as np

if not (has_lidar and has_calib):
    print("LiDAR data not found. To use this section:")
    print("  1. Upload sensorfusion.zip to Colab")
    print("  2. Run: !unzip sensorfusion.zip -d /content/")
    print("  3. Re-run the data loading cell above")
else:
    images = sorted(glob.glob(os.path.join(IMAGE_DIR, "*.png")))[:3]
    os.makedirs("outputs/examples", exist_ok=True)
    
    for img_path in images:
        frame_id = os.path.splitext(os.path.basename(img_path))[0]
        vel_path = os.path.join(VELODYNE_DIR, f"{frame_id}.bin")
        cal_path = os.path.join(CALIB_DIR, f"{frame_id}.txt")
        
        if not (os.path.exists(vel_path) and os.path.exists(cal_path)):
            print(f"  Skipping {frame_id} — missing velodyne or calib")
            continue
        
        print(f"\nProcessing {frame_id}...")
        
        # Project LiDAR onto image
        overlay_path = f"outputs/examples/{frame_id}_overlay.png"
        overlay = create_lidar_overlay(img_path, vel_path, cal_path, save_path=overlay_path)
        
        # Depth-aware VLM analysis
        if overlay:
            print("  Running depth-aware analysis...")
            depth_result = analyze_scene(overlay, model, processor, prompt_type="depth_aware")
            print(f"  {depth_result[:200]}...\n")
            display_results(overlay, depth_result, title=f"Depth-Aware: {frame_id}")
        
        # 3-panel combined view
        combined_path = f"outputs/examples/{frame_id}_combined.png"
        create_combined_view(img_path, vel_path, cal_path, save_path=combined_path)
        
        torch.cuda.empty_cache()

## Bird's Eye View

A top-down projection of the LiDAR point cloud. Each bright pixel is a laser return. You can clearly make out road boundaries, parked vehicles, and building facades. I'm using a 40m x 40m window at 0.1m per pixel resolution.

In [ ]:
if has_lidar:
    vel_files = sorted(glob.glob(os.path.join(VELODYNE_DIR, "*.bin")))[:3]
    
    fig, axes = plt.subplots(1, len(vel_files), figsize=(6 * len(vel_files), 6))
    if len(vel_files) == 1:
        axes = [axes]
    
    for ax, vel_path in zip(axes, vel_files):
        frame_id = os.path.splitext(os.path.basename(vel_path))[0]
        bev = create_bev(vel_path)
        ax.imshow(bev, cmap='hot', origin='lower')
        ax.set_title(f"BEV: {frame_id}", fontsize=11)
        ax.set_xlabel('Lateral (m)')
        ax.set_ylabel('Longitudinal (m)')
    
    plt.suptitle("Bird's Eye View from Velodyne LiDAR", fontsize=13)
    plt.tight_layout()
    plt.show()
else:
    print("No Velodyne data available for BEV visualization.")

## Try Your Own Image

Upload any driving scene — highway, rural road, construction zone, parking lot — and see how the VLM handles it. This is the real test of zero-shot capability.

A YOLO model trained on KITTI's 8 classes will output "unknown" for a fallen tree, a horse cart, or a flooded road. The VLM describes what it sees, assesses the risk, and tells you what to do. That's the paradigm shift.

In [ ]:
from google.colab import files

print("Upload a driving scene image (PNG or JPG):")
uploaded = files.upload()

if uploaded:
    for filename in uploaded.keys():
        image = Image.open(filename).convert("RGB")
        print(f"\nAnalyzing: {filename} ({image.size[0]}x{image.size[1]})\n")
        
        result = analyze_scene(image, model, processor, prompt_type="full_analysis")
        
        print(result)
        display_results(image, result, title=f"Custom Scene: {filename}")
        torch.cuda.empty_cache()

## Tests

11 unit tests covering LiDAR loading, calibration parsing, projection math, prompt validation, and model config. All run without a GPU.

In [ ]:
!pip install -q pytest
!python -m pytest tests/ -v --tb=short

## What I Learned Building This

**Prompt engineering matters more than model size.** The gap between a generic "describe this image" and a well-structured ADAS prompt is enormous. Asking the model to organize its output into scene context, objects, hazards, and recommendations made the responses dramatically more useful and consistent.

**4-bit quantization is surprisingly good.** I expected visible quality loss going from fp16 to 4-bit NF4. In practice, the scene descriptions are nearly identical. The VRAM savings (14GB down to 5GB) make this accessible on consumer GPUs and free Colab instances.

**LiDAR fusion through the visual channel works.** Projecting depth-colored points onto the image and letting the VLM interpret the colors is a simple but effective way to inject 3D awareness without modifying the model architecture. The model's distance estimates become noticeably more precise with the overlay.

**VLMs complement traditional detectors, they don't replace them.** For safety-critical real-time ADAS, you still want a YOLO or PointPillars at 30+ FPS for core detection. The VLM adds something those models fundamentally cannot: semantic scene understanding, handling of novel objects, and natural language output for situations that fall outside the training distribution.

That last point is really the thesis of this project. Traditional ADAS perception is excellent at the things it's been trained on and completely blind to everything else. VLMs bridge that gap — and as these models get faster and smaller, I think they'll become a standard component in production perception stacks.

---

**Author:** Vasu Tammisetti — AI Research Engineer & Doctoral Researcher, Infineon Technologies AG, Munich

**GitHub:** [VLM-LiDAR-Camera-ADAS-perception](https://github.com/VasuTammisetti/VLM-LiDAR-Camera-ADAS-perception) | **Model:** [LLaVA-1.6-Mistral-7B](https://huggingface.co/llava-hf/llava-v1.6-mistral-7b-hf) | **Dataset:** [KITTI](http://www.cvlibs.net/datasets/kitti/)